## Para utilizar en la clase 03, del 30-mayo-2025
##  **NO UTILIZAR ANTES**

# AutoGluon
Matar una mosca con una  bazooka, entrena automaticamente modelos de:


*   Estadistica Clasica
*   Machine Learning
*   Deep Learning


 ['SeasonalNaive', 'RecursiveTabular', 'DirectTabular', 'DynamicOptimizedTheta', 'Chronos2', 'Chronos2SmallFineTuned', 'AutoETS', 'ChronosWithRegressor[bolt_small]', 'TemporalFusionTransformer', 'DeepAR']



## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [31]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [32]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


# 1  Modelo AutoGluon

## 1.1 Init Experimento

In [33]:
# instalacion de paquetes que NO vienen por default en Colab
# autogluon es MUY pesado. varios minutos se perderan aqui
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]


Using Python 3.12.13 environment at: /usr
Checked 1 package in 152ms


In [34]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [35]:
import os as os
import numpy as np
import polars as pl

from datetime import datetime
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

Por favor, cargar aqui SU semilla primigenia
<br> **Muy importante**, cambiar el numero de experimento en cada corrida. Usted ha sido notificado !
<br> Si cada corrida no está en una nueva carpeta virgen, entonces se reutilizarn modelos viejos corridos con otros parametros
https://www.youtube.com/shorts/0_0Kzqpdn1o

In [36]:
# defino los parametros
PARAM = {'experimento':'AutoGluon-01',
  'kaggle_competition':'labo-iii-2026-rosario',
  'semilla_primigenia':102191
}

In [37]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AutoGluon-01


## 1.2 Init AutoGluon

In [38]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [39]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [40]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [41]:
# paso de periodo a  timestamp
tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

Opcion de *empiojar el dataset*
<br>agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [42]:
empiojar= False
empiojar_ruido= 0.42

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


## 1.3 Entrenamiento AutoGluon

La magia del Auto Machine Learning  aplicada a un dataset que posee MULTIPLES series de tiempo que se analizan en forma conjunta, suponiendo algun tipo de correlacion entre ellas.

AutoGluon TimeSeriesPredictor predicts future values of **multiple related** time series.

TimeSeriesPredictor provides probabilistic (quantile) multi-step-ahead orecasts for univariate time series. The forecast includes both the mean (i.e.,
 onditional expectation of future values given the past), as well as the quantiles of the forecast distribution, indicating the range of possible future outcomes.

TimeSeriesPredictor fits both “global” deep learning models that are **shared across all time series** (e.g., DeepAR, Transformer), as well as “local”
 statistical models that are fit to each individual time series (e.g., ARIMA, ETS).

TimeSeriesPredictor expects input data and makes predictions in the TimeSeriesDataFrame format.


https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [43]:
# paso a formato  ts = time_series
ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas.to_pandas(), # tristemente AutoGluon espera un dataframe Pandas ...
  timestamp_column='timestamp',
  id_column='product_id'
)

Elegir cuidadosamente la metrica a utilizar
<br>Probar alternativas !
<br>Ese celda lleva 30 minutos en correr
<br>https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-metrics.html#forecasting-metrics
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.fit.html#autogluon.timeseries.TimeSeriesPredictor.fit
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [44]:
# Entrenamiento, esta es la parte pesada
global_eval_metric = 'RMSE' # alternativa RMSE

# defino
modelo = TimeSeriesPredictor(
  prediction_length= 2,  # horizonte de prediccion
  target= 'tn',
  freq= 'MS',  # Frecuencia mensual (Month Start)
  eval_metric= global_eval_metric
)

# entreno, le tira con muchisimos algorimtos, y prueba ensamblarlos
modelo.fit(ts_data,
  num_val_windows= 2,
  time_limit= 3600, # una hora
  presets= "best_quality",  # Máxima calidad posible, obvio mayor tiempo de corrida
  random_seed= PARAM['semilla_primigenia']
)

Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/content/.drive/My Drive/labo3/exp/AutoGluon-01/AutogluonModels/ag-20260530_145607'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       10.36 GB / 12.67 GB (81.7%)
Disk Space Avail:   74.86 GB / 107.72 GB (69.5%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed':

## 1.4 Prediccion AutoGluon

In [45]:
# predict a partir los mismos datos

tb_forecast = modelo.predict(ts_data,
  random_seed= PARAM['semilla_primigenia']
)

display(tb_forecast)

data with frequency 'IRREG' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


mean         0.1          0.2          0.3  \
item_id timestamp                                                       
20001   2020-01-01  1288.720851  866.709064  1025.883245  1119.704590   
        2020-02-01  1354.673445  927.723942  1082.946200  1169.064014   
20002   2020-01-01  1179.350112  844.625653   975.293721  1053.661560   
        2020-02-01  1131.507596  758.726742   888.065407   970.372050   
20003   2020-01-01   791.583079  573.454649   650.202746   704.543501   
...                         ...         ...          ...          ...   
21266   2020-02-01     0.037125   -0.070824    -0.032614    -0.005240   
21267   2020-01-01     0.019272   -0.032517    -0.015084    -0.002058   
        2020-02-01     0.017271   -0.057092    -0.029433    -0.011680   
21276   2020-01-01     0.010104   -0.009540    -0.003247     0.002630   
        2020-02-01     0.009662   -0.015904    -0.006714    -0.000356   

                            0.4          0.5          0.6          0.7  \
item_id timestamp                                                        
20001   2020-01-01  1203.388941  1287.647400  1348.199432  1450.115824   
        2020-02-01  1254.633680  1346.891350  1434.650590  1516.704601   
20002   2020-01-01  1119.471946  1179.317115  1248.729647  1320.182192   
        2020-02-01  1053.879464  1125.157989  1198.458739  1269.651955   
20003   2020-01-01   745.966914   794.164752   837.944418   886.820956   
...                         ...          ...          ...          ...   
21266   2020-02-01     0.016662     0.037038     0.056768     0.079656   
21267   2020-01-01     0.008654     0.018653     0.028822     0.040485   
        2020-02-01     0.002992     0.016448     0.031300     0.045645   
21276   2020-01-01     0.006344     0.009478     0.013463     0.017089   
        2020-02-01     0.004218     0.009621     0.015214     0.020201   

                            0.8          0.9  
item_id timestamp                             
20001   2020-01-01  1568.108005  1722.272851  
        2020-02-01  1616.987590  1746.022519  
20002   2020-01-01  1413.189267  1534.667932  
        2020-02-01  1367.813980  1523.626468  
20003   2020-01-01   926.860101  1016.494218  
...                         ...          ...  
21266   2020-02-01     0.106731     0.146960  
21267   2020-01-01     0.053558     0.071299  
        2020-02-01     0.062793     0.090999  
21276   2020-01-01     0.021560     0.030952  
        2020-02-01     0.026925     0.037287  

[1560 rows x 10 columns]

In [46]:
# paso a formato Polars, teniendo en cuenta el indice
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

In [47]:
# me quedo con ls predicciones de febrero-2020
# en 'mean' esta la prediccion, mas alla de los n-tiles
tb_final = tb_forecast.filter(pl.col("timestamp") ==  datetime(2020, 2, 1)).select(["item_id","mean"])

display(tb_final)

item_id,mean
i64,f64
20001,1354.673445
20002,1131.507596
20003,737.749279
20004,551.613523
20005,546.519282
…,…
21263,0.036332
21265,0.036246
21266,0.037125


## 1.5 Submit a Kaggle

In [48]:
# cambio nombre de campos a los que reconoce Kaggle
tb_final = tb_final.rename({
  "item_id": "product_id",
  "mean": "tn",
})

In [49]:
# Submit a Kaggle
if not empiojar:
  archivo= "AutoGluon_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon " + global_eval_metric
else:
  archivo= "AutoGluon_empiojado_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon logEMPIOJADO " + global_eval_metric + " al " + str(empiojar_ruido)

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )